# Geotagger — service notebook

Hits the deployed `/geotag` endpoint (proxied at `wiig.dia.fi.upm.es/nlp`) with `debug=true`
and shows **every internal stage** per test case:

1. **spans** — raw NER + street-regex detection
2. **typed** — the type assigned to each span (regex / gazetteer feature class / nli fallback)
3. **detected_cities** — the cities used to impute streets & locations
4. **places** — the final resolved output

Test cases live in `eval/fixtures/geotagger_cases.json`.

In [ ]:
import os, json, requests
from pathlib import Path

NLP_BASE_URL = os.environ.get('NLP_BASE_URL', 'https://wiig.dia.fi.upm.es/nlp').rstrip('/')
ENDPOINT     = f'{NLP_BASE_URL}/geotag'
HEADERS      = {'Content-Type': 'application/json'}
TIMEOUT      = 120

CASES = json.loads(Path('fixtures/geotagger_cases.json').read_text(encoding='utf-8'))
print(f'{len(CASES)} cases -> {ENDPOINT}')

In [ ]:
def call_geotag(inp):
    body = {**inp, 'debug': True}
    r = requests.post(ENDPOINT, json=body, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()

def _row(d, keys):
    return '  '.join(f'{k}={d.get(k)!r}' for k in keys)

def render(resp):
    tr = resp.get('trace') or {}
    print('  STAGE 1 · spans (NER + regex)')
    for s in tr.get('spans', []):
        print('     ', _row(s, ['text', 'label', 'hint']))
    print('  STAGE 2 · typed')
    for s in tr.get('typed', []):
        print('     ', _row(s, ['text', 'type']))
    print('  STAGE 3 · detected_cities (imputation context)')
    for c in tr.get('detected_cities', []):
        print('     ', _row(c, ['city_id', 'city_name', 'lat', 'lon']))
    print('  FINAL · places')
    for p in resp.get('places', []):
        print('     ', _row(p, ['text', 'type', 'name', 'city_id', 'city_name', 'edge_ids', 'lat', 'lon']))

In [ ]:
def _matches(exp, p):
    if p.get('type') != exp['type']:
        return False
    if 'name' in exp:
        n = exp['name'].lower()
        if (p.get('name') or '').lower() != n and (p.get('city_name') or '').lower() != n:
            return False
    if 'city_name' in exp:
        if (p.get('city_name') or '').lower() != exp['city_name'].lower():
            return False
    return True

def score(expected, places):
    if not expected:  # national-scope case: pass when no city/region resolved
        bad = [p for p in places if p.get('type') in ('city', 'region')]
        return (1, 1) if not bad else (0, 1)
    matched = sum(1 for exp in expected if any(_matches(exp, p) for p in places))
    return matched, len(expected)

In [ ]:
results = []
for case in CASES:
    print(f"── {case['id']}: {case['description']}")
    try:
        resp = call_geotag(case['input'])
    except Exception as exc:
        print('   ERROR:', exc); print(); results.append((case['id'], 0, 1)); continue
    render(resp)
    m, total = score(case.get('expected_places', []), resp.get('places', []))
    icon = '✅' if m == total else '❌'
    print(f'   {icon} matched {m}/{total} expected places')
    print()
    results.append((case['id'], m, total))

In [ ]:
passed = sum(1 for _, m, t in results if m == t)
tm = sum(m for _, m, _ in results); tt = sum(t for _, _, t in results)
print('SCORECARD · /geotag')
print(f'  Cases passing : {passed}/{len(results)}')
print(f'  Places matched: {tm}/{tt}')
for cid, m, t in results:
    print(f"   {'✅' if m==t else '❌'} {cid}: {m}/{t}")